In [1]:
from google.colab import drive
drive.mount('/content/drive')

import glob
BASE = '/content/drive/MyDrive/deepfake_v2'

real  = glob.glob(f'{BASE}/roop_input_v2/*.mp4')
swap  = glob.glob(f'{BASE}/fake_full_v2/*.mp4')
ai    = glob.glob(f'{BASE}/fake_ai_generated/*.mp4')

print(f"Real videos     : {len(real)}")
print(f"Face-swap fakes : {len(swap)}")
print(f"AI-generated    : {len(ai)}")

Mounted at /content/drive
Real videos     : 92
Face-swap fakes : 92
AI-generated    : 30


In [2]:
import os, cv2, glob
import numpy as np
from sklearn.model_selection import train_test_split

BASE     = '/content/drive/MyDrive/deepfake_v2'
AI_DIR   = f'{BASE}/fake_ai_generated'
SAVE_DIR = f'{BASE}/deepfake_processed_v4'
os.makedirs(SAVE_DIR, exist_ok=True)

FRAMES   = 15
IMG_SIZE = 224

detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    'haarcascade_frontalface_default.xml'
)

def extract_faces(path, n=15):
    cap   = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < n:
        cap.release()
        return None
    indices = np.linspace(0, total-1, n, dtype=int)
    faces   = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        dets = detector.detectMultiScale(
            gray, scaleFactor=1.1,
            minNeighbors=4, minSize=(50,50)
        )
        if len(dets) > 0:
            dets = sorted(dets,
                key=lambda x: x[2]*x[3], reverse=True)
            x,y,w,h = dets[0]
            face = frame[y:y+h, x:x+w]
            face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
            face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
            faces.append(face.astype(np.float32)/255.0)
        else:
            faces.append(
                np.zeros((IMG_SIZE,IMG_SIZE,3), np.float32))
    cap.release()
    if len(faces) < n:
        return None
    return np.array(faces[:n])

# Load existing v3 data (real=0, face-swap=1)
print("Loading existing v3 data...")
X_exist = np.concatenate([
    np.load(f'{BASE}/deepfake_processed_v3/X_train_seq.npy'),
    np.load(f'{BASE}/deepfake_processed_v3/X_val_seq.npy'),
    np.load(f'{BASE}/deepfake_processed_v3/X_test_seq.npy'),
])
y_exist = np.concatenate([
    np.load(f'{BASE}/deepfake_processed_v3/y_train_seq.npy'),
    np.load(f'{BASE}/deepfake_processed_v3/y_val_seq.npy'),
    np.load(f'{BASE}/deepfake_processed_v3/y_test_seq.npy'),
])
print(f"Existing: {len(X_exist)} seqs — "
      f"Real={sum(y_exist==0)} FaceSwap={sum(y_exist==1)}")

# Extract AI-generated → label 2
ai_seqs, ai_labels = [], []
ai_list = sorted(glob.glob(f'{AI_DIR}/*.mp4'))
print(f"\nProcessing {len(ai_list)} AI-generated videos...")

for i, p in enumerate(ai_list):
    s = extract_faces(p, FRAMES)
    if s is not None:
        ai_seqs.append(s)
        ai_labels.append(2)
        print(f"  [{i+1}/{len(ai_list)}] OK: {os.path.basename(p)}")
    else:
        print(f"  [{i+1}/{len(ai_list)}] SKIP: {os.path.basename(p)}")

print(f"\nAI extracted: {len(ai_seqs)}")

Loading existing v3 data...
Existing: 184 seqs — Real=92 FaceSwap=92

Processing 30 AI-generated videos...
  [1/30] OK: sinhala01.mp4
  [2/30] OK: sinhala02.mp4
  [3/30] OK: sinhala03.mp4
  [4/30] OK: sinhala04.mp4
  [5/30] OK: sinhala05.mp4
  [6/30] OK: sinhala06.mp4
  [7/30] OK: sinhala07.mp4
  [8/30] OK: sinhala08.mp4
  [9/30] OK: sinhala09.mp4
  [10/30] OK: sinhala10.mp4
  [11/30] SKIP: sinhala11.mp4
  [12/30] SKIP: sinhala12.mp4
  [13/30] OK: sinhala13.mp4
  [14/30] OK: sinhala14.mp4
  [15/30] OK: sinhala15.mp4
  [16/30] OK: sinhala16.mp4
  [17/30] OK: tamil01.mp4
  [18/30] OK: tamil02.mp4
  [19/30] OK: tamil03.mp4
  [20/30] OK: tamil04.mp4
  [21/30] OK: tamil05.mp4
  [22/30] OK: tamil06.mp4
  [23/30] OK: tamil07.mp4
  [24/30] OK: tamil08.mp4
  [25/30] OK: tamil09.mp4
  [26/30] OK: tamil10.mp4
  [27/30] SKIP: tamil11.mp4
  [28/30] OK: tamil12.mp4
  [29/30] OK: tamil13.mp4
  [30/30] OK: tamil14.mp4

AI extracted: 27


In [3]:
# Combine existing + AI
X_all = np.concatenate([
    X_exist,
    np.array(ai_seqs, dtype=np.float32)
])
y_all = np.concatenate([
    y_exist,
    np.array(ai_labels, dtype=np.int32)
])

print(f"Total     : {len(X_all)}")
print(f"Real      : {sum(y_all==0)}")
print(f"FaceSwap  : {sum(y_all==1)}")
print(f"AI-gen    : {sum(y_all==2)}")

# Split
X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all,
    test_size=0.15,
    stratify=y_all,
    random_state=42
)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_tr, y_tr,
    test_size=0.15,
    stratify=y_tr,
    random_state=42
)

print(f"\nTrain : {len(X_tr)}")
print(f"Val   : {len(X_va)}")
print(f"Test  : {len(X_te)}")

# Save
np.save(f'{SAVE_DIR}/X_train_seq.npy', X_tr)
np.save(f'{SAVE_DIR}/X_val_seq.npy',   X_va)
np.save(f'{SAVE_DIR}/X_test_seq.npy',  X_te)
np.save(f'{SAVE_DIR}/y_train_seq.npy', y_tr)
np.save(f'{SAVE_DIR}/y_val_seq.npy',   y_va)
np.save(f'{SAVE_DIR}/y_test_seq.npy',  y_te)

print(f"\nSaved to: {SAVE_DIR}")

# Verify
for name, arr in [('y_train',y_tr),('y_val',y_va),('y_test',y_te)]:
    print(f"{name}: shape={arr.shape} "
          f"unique={np.unique(arr, return_counts=True)}")

Total     : 211
Real      : 92
FaceSwap  : 92
AI-gen    : 27

Train : 152
Val   : 27
Test  : 32

Saved to: /content/drive/MyDrive/deepfake_v2/deepfake_processed_v4
y_train: shape=(152,) unique=(array([0, 1, 2], dtype=int32), array([66, 66, 20]))
y_val: shape=(27,) unique=(array([0, 1, 2], dtype=int32), array([12, 12,  3]))
y_test: shape=(32,) unique=(array([0, 1, 2], dtype=int32), array([14, 14,  4]))
